In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. Install Ollama from https://ollama.com/download for your operating system.\n   - macOS: download and install the `.pkg`\n   - Windows: download and install the `.msi`\n   - Linux: run:\n   ```bash\n   curl -fsSL https://ollama.com/install.sh | sh\n   ```\n\n2. Open a terminal and start a model:\n```bash\nollama run llama3\n```\n\nThis will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\n3. To test that the local server is running, you can use:\n```bash\ncurl http://localhost:11434\n```\n\nIf you get a connection refused error, restart the Ollama server with:\n```bash\n!nohup ollama serve > nohup.out 2>&1 &\n```'

In [5]:
assistant.rag("How do I run Olama locally?")

'The FAQ only mentions running the **course locally**, not **Olama** specifically.\n\nIf you mean running the course environment locally, the FAQ says:\n\n- Yes, you can run the course locally instead of using Codespaces.\n- You should be comfortable setting up:\n  - Python\n  - `uv`\n  - Jupyter\n  - Docker\n  - any other tools needed for the module\n- If you run locally, document your setup and keep it reproducible.\n\nIf you meant a different tool by “Olama,” I don’t have any FAQ entry for that.'

In [6]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—usually you can join a course after it starts, but it depends on the course rules and whether registration is still open.\n\nA few things to check:\n- **Enrollment deadline:** some courses allow late registration, others don’t.\n- **Prerequisites / approval:** you may need instructor or department permission.\n- **Missed material:** if it’s already underway, you may need to catch up on lectures or assignments.\n- **Capacity:** the course may be full.\n\nIf you want, I can help you draft a short message to the instructor or registrar asking whether late enrollment is possible.'

In [7]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [9]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join after the course has started? discovered the course join late enrollment"}', call_id='call_LQXlOvz12X7K8tjYuKVUp18A', name='search', type='function_call', id='fc_0b65d985211b2f35006a4cab6b56648198b1b6b1fee65b7a09', namespace=None, status='completed')]

In [10]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [11]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [12]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join and start learning anytime.\n\nIf you want a certificate, make sure to submit your project while submissions are still being accepted.'

In [13]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(838, 34)

In [14]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [15]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [16]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [17]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment registration late join"}
function_call: search {"query":"new student can I join course after start enrollment FAQ"}


In [18]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you’ll need to submit your project while submissions are still open. If you’re only following along for learning, you can start anytime.

If you’d like, I can also help with how to start the course or explain the certificate rules. Any other areas you want to explore?


In [19]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [20]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run install local Ollama"}
function_call: search {"query":"Ollama local run model localhost setup"}
function_call: search {"query":"Olama Ollama FAQ local"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - **macOS**: download and install from https://ollama.com/download
   - **Windows**: download the `.msi` from the same page
   - **Linux**:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a local model**
   ```bash
   ollama run llama3
   ```
   This downloads the model and starts a local chat session.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response from the Ollama server.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )

   p

'To run Ollama locally:\n\n1. **Install Ollama**\n   - **macOS**: download and install from https://ollama.com/download\n   - **Windows**: download the `.msi` from the same page\n   - **Linux**:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a local model**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model and starts a local chat session.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response from the Ollama server.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a **connection refused** error, restart the server with:\n```bash\nollama serve\n```\nor, in a notebook:\n```bash\n!nohup ollama serve > nohup.ou

In [21]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late enrollment still join discovered course can I still join"}
function_call: search {"query":"course FAQ late enrollment registration join after course starts discovered course"}
function_call: search {"query":"can I still join the course if I just discovered it FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, make sure you submit your project while submissions are still being accepted. You can also start learning and submitting homework as long as the submission forms are open.

If you'd like, I can also explain the best way to catch up quickly or how certificates work. Any other areas you want to explore?


"Yes — you can still join the course.\n\nIf you want a certificate, though, make sure you submit your project while submissions are still being accepted. You can also start learning and submitting homework as long as the submission forms are open.\n\nIf you'd like, I can also explain the best way to catch up quickly or how certificates work. Any other areas you want to explore?"

In [22]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [23]:
agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment late registration access FAQ"}
iteration #2...
ASSISTANT:
Yes — you can join the course even if you just discovered it.

According to the course FAQ, you’re accepted and can start learning right away. You can also submit homework while the form is open, and registration is only used to gauge interest.

One important note: if you want a certificate, you need to submit your project while submissions are still being accepted.

If you want, I can also help you with what to do next after joining, or explain the certificate requirements.


'Yes — you can join the course even if you just discovered it.\n\nAccording to the course FAQ, you’re accepted and can start learning right away. You can also submit homework while the form is open, and registration is only used to gauge interest.\n\nOne important note: if you want a certificate, you need to submit your project while submissions are still being accepted.\n\nIf you want, I can also help you with what to do next after joining, or explain the certificate requirements.'

In [24]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit what is it"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening move 1 d4 d5 c4 FAQ"}
iteration #3...
function_call: search {"query":"gambit chess opening what is a gambit sacrifice pawn queen's gambit"}
iteration #4...
ASSISTANT:
The **Queen’s Gambit** is a chess opening that starts with:

1. **d4 d5**
2. **c4**

White offers the c-pawn to try to draw Black’s d-pawn away from the center. It’s called a **gambit** because White is offering a pawn in exchange for better control of the center and more active play.

Common ideas:
- White wants to pressure Black’s center
- Black can accept or decline the pawn offer
- It’s one of the oldest and most respected chess openings

There are two main branches:
- **Queen’s Gambit Accepted**: Black takes the pawn on c4
- **Queen’s Gambit Declined**: Black does not take it, usually playing ...e6 or ...c6

If you want, I can also explain:

'The **Queen’s Gambit** is a chess opening that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the c-pawn to try to draw Black’s d-pawn away from the center. It’s called a **gambit** because White is offering a pawn in exchange for better control of the center and more active play.\n\nCommon ideas:\n- White wants to pressure Black’s center\n- Black can accept or decline the pawn offer\n- It’s one of the oldest and most respected chess openings\n\nThere are two main branches:\n- **Queen’s Gambit Accepted**: Black takes the pawn on c4\n- **Queen’s Gambit Declined**: Black does not take it, usually playing ...e6 or ...c6\n\nIf you want, I can also explain:\n- the opening moves in simple terms,\n- the difference between accepted and declined,\n- or why it’s considered such a strong opening.'

In [25]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queens gambit chess opening queen's gambit"}
iteration #3...
ASSISTANT:
I couldn’t find a course/FAQ entry for “queen gambit,” so this looks off-topic for the course database.

If you meant the chess opening “Queen’s Gambit,” I can’t answer it from the FAQ here. If you have a course-related question, send it over and I’ll check. Are there other areas you want to explore?


'I couldn’t find a course/FAQ entry for “queen gambit,” so this looks off-topic for the course database.\n\nIf you meant the chess opening “Queen’s Gambit,” I can’t answer it from the FAQ here. If you have a course-related question, send it over and I’ll check. Are there other areas you want to explore?'

In [26]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [27]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [28]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [29]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [30]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [31]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [32]:
result.cost

CostInfo(input_cost=Decimal('0.00273525'), output_cost=Decimal('0.001467'), total_cost=Decimal('0.00420225'))

In [33]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run Ollama local install run locally FAQ"}', cal

In [34]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [ ]:
runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


...

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received
